# 使用LM-Studio來追蹤OpenAI API如何處理Tools

In [1]:
from langchain_openai import ChatOpenAI
model_name = 'google/gemma-3-4b'  # 指定模型名稱，模型名稱會根據下載的模型不同而改變

base_url = 'http://localhost:1234/v1'  # LM Studio 本地伺服器的URL
llm = ChatOpenAI(
    model=model_name,
    openai_api_key="not-needed",
    openai_api_base=base_url 
)

for chunk in llm.stream("一句話說明機器學習的定義"):
    print(chunk.content, end='')

機器學習是指讓電腦從數據中學習，而無需明確編程，使其能夠預測、分類或做出決策。


In [2]:
from langchain_core.tools import tool # <-- 這裡是新增的

# 匯入 geopy 套件中的 Nominatim 類別，用於地理編碼（將地名轉換成經緯度）
from geopy.geocoders import Nominatim

# 定義函式：輸入城市名稱，回傳其經緯度座標
@tool # <-- 這裡是新增的
def tool_get_coordinates(city_name:str):
    """取得城市GPS座標，先用 Nominatim，失敗時改用 ArcGIS"""

    # ---- 第一來源：Nominatim ----
    try:
        geolocator1 = Nominatim(user_agent="clement_fallback_test")
        location = geolocator1.geocode(city_name, timeout=10)

        if location:
            return (location.latitude, location.longitude)
    except:
        pass  # 忽略錯誤，直接進入第二來源

    # ---- 第二來源：ArcGIS  ----
    try:
        geolocator2 = ArcGIS(timeout=10)
        location = geolocator2.geocode(city_name)

        if location:
            return (location.latitude, location.longitude)
    except Exception as e:
        return e

    return None

# 匯入 requests 套件，用於發送 HTTP 請求
import requests

# 定義函式：輸入經緯度，回傳目前天氣資訊（溫度）
@tool # <-- 這裡是新增的
def tool_get_weather(latitude:float, longitude:float):
    """取得溫度值

    Args:
        latitude: GPS經度
        longitude: GPS緯度
    """    

    # 使用 Open-Meteo API 發送 GET 請求，取得氣象資料
    # API 參數：
    # - latitude / longitude: 經緯度
    # - current: 取得目前時刻的溫度 (temperature_2m) 與風速 (wind_speed_10m)
    # - hourly: 取得每小時溫度、相對濕度、風速
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={latitude}&longitude={longitude}&"
        f"current=temperature_2m,wind_speed_10m&"
        f"hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    
    # 將回傳的 JSON 資料解析成 Python 字典
    data = response.json()
    
    # 回傳目前時刻的氣溫（單位：攝氏度）
    return data['current']['temperature_2m']

In [3]:
# 工具列表
tools = [tool_get_coordinates, tool_get_weather]

# 連結工具到LLM
llm_with_tools = llm.bind_tools(tools)

In [4]:
from langchain_core.messages import HumanMessage

# 使用者輸入訊息，詢問台北市目前的溫度
messages = [
    HumanMessage("台北市現在的溫度？")
]

# 透過 invoke() 呼叫 LLM，並傳入訊息列表
response_01 = llm_with_tools.invoke(messages)

# 查看 LLM 的回應
response_01


AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 557, 'total_tokens': 608, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'google/gemma-3-4b', 'system_fingerprint': 'google/gemma-3-4b', 'id': 'chatcmpl-h0xedp0ibmtjz02ozasfe', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e2ab9-1309-7762-8cbb-3d27541157aa-0', tool_calls=[{'name': 'get_weather', 'args': {'latitude': 25.0479, 'longitude': 121.4863}, 'id': '792919108', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 557, 'output_tokens': 51, 'total_tokens': 608, 'input_token_details': {}, 'output_token_details': {}})

## 查看LM-Studio的log

### Langchain LLM 傳遞過去的內容
```
2026-05-15 16:20:34 [DEBUG]
 Received request: POST to /v1/chat/completions with body  {
  "messages": [
    {
      "content": "台北市現在的溫度？",
      "role": "user"
    }
  ],
  "model": "google/gemma-3-4b",
  "stream": false,
  "tools": [
    {
      "type": "function",
      "function": {
        "name": "tool_get_coordinates",
        "description": "取得城市GPS座標，先用 Nominatim，失敗時改用 ArcGIS",
        "parameters": {
          "properties": {
            "city_name": {
              "type": "string"
            }
          },
          "required": [
            "city_name"
          ],
          "type": "object"
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "tool_get_weather",
        "description": "取得溫度值\n\nArgs:\n    latitude: GPS經度\n    longitude: GPS緯度",
        "parameters": {
          "properties": {
            "latitude": {
              "type": "number"
            },
            "longitude": {
              "type": "number"
            }
          },
          "required": [
            "latitude",
            "longitude"
          ],
          "type": "object"
        }
      }
    }
  ]
}
```

### LM-Studio回傳的內容
```
2026-05-15 16:20:59  [INFO]
 [google/gemma-3-4b] Generated prediction:  {
  "id": "chatcmpl-h0xedp0ibmtjz02ozasfe",
  "object": "chat.completion",
  "created": 1778833234,
  "model": "google/gemma-3-4b",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "",
        "tool_calls": [
          {
            "type": "function",
            "id": "792919108",
            "function": {
              "name": "get_weather",
              "arguments": "{\"latitude\":25.0479,\"longitude\":121.4863}"
            }
          }
        ]
      },
      "logprobs": null,
      "finish_reason": "tool_calls"
    }
  ],
  "usage": {
    "prompt_tokens": 557,
    "completion_tokens": 51,
    "total_tokens": 608
  },
  "stats": {},
  "system_fingerprint": "google/gemma-3-4b"
}
```